# 🔧 Bank Marketing - Data Preprocessing

In this notebook, we prepare the Bank Marketing dataset
for machine learning models by:
- Cleaning the data
- Encoding categorical variables
- Scaling numerical features
- Balancing the dataset using SMOTE

**Input:**  bank-full.csv (45,211 rows × 17 columns)
**Output:** Clean, ready-to-train dataset

#Install & Import Libraries

In [31]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 📂 Load Dataset

Loading the bank-full.csv dataset that we analyzed in the EDA notebook.

In [32]:
# Load dataset
df = pd.read_csv('bank-full.csv')

print("✅ Dataset loaded successfully!")
print("Shape:", df.shape)

✅ Dataset loaded successfully!
Shape: (45211, 17)


## 🔍 Step 1: Check Duration = 0

Duration represents the call duration in seconds.
If Duration = 0, it means no real call was made.
We need to check how many rows have Duration = 0
before deciding what to do.

In [33]:
# Check Duration = 0
print("Duration = 0 rows:", (df['duration'] == 0).sum())
print("Total rows:", len(df))
print("Percentage:", round((df['duration'] == 0).sum() / len(df) * 100, 2), "%")

Duration = 0 rows: 3
Total rows: 45211
Percentage: 0.01 %


## 🗑️ Step 2: Remove Duration = 0 Rows

Only 3 rows have Duration = 0 (0.01% of data).
This is a very small number, so we can safely remove them
without losing important information.

In [34]:
# Remove Duration = 0 rows
df = df[df['duration'] != 0]

print("✅ Removed Duration = 0 rows!")
print("New Shape:", df.shape)
print("Rows removed:", 45211 - len(df))

✅ Removed Duration = 0 rows!
New Shape: (45208, 17)
Rows removed: 3


## 🔧 Step 3: Fix Pdays = -1

Pdays represents the number of days since the client
was last contacted from a previous campaign.
-1 means the client was never contacted before.
We will replace -1 with 0 for better interpretation.

In [35]:
# Fix Pdays = -1 → 0
print("Before - Pdays = -1:", (df['pdays'] == -1).sum())

df['pdays'] = df['pdays'].replace(-1, 0)

print("After  - Pdays = -1:", (df['pdays'] == -1).sum())
print("After  - Pdays = 0 :", (df['pdays'] == 0).sum())
print("✅ Pdays fixed successfully!")

Before - Pdays = -1: 36951
After  - Pdays = -1: 0
After  - Pdays = 0 : 36951
✅ Pdays fixed successfully!


## 🔄 Step 4: Encode Categorical Variables

Converting categorical variables to numerical values:

- Binary columns (yes/no)  → Label Encoding (0/1)
- Multi-value columns      → One-Hot Encoding
- Target column (y)        → Label Encoding (yes=1 / no=0)

In [36]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# 1️⃣ Binary columns → Label Encoding
binary_cols = ['default', 'housing', 'loan']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
print("✅ Binary columns encoded!")

# 2️⃣ Target column → Label Encoding
df['y'] = le.fit_transform(df['y'])
print("✅ Target column encoded!")

# 3️⃣ Multi-value columns → One-Hot Encoding
multi_cols = ['job', 'marital', 'education',
              'contact', 'month', 'poutcome']
df = pd.get_dummies(df, columns=multi_cols)
print("✅ Multi-value columns encoded!")

print("\nNew Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

✅ Binary columns encoded!
✅ Target column encoded!
✅ Multi-value columns encoded!

New Shape: (45208, 49)

Columns: ['age', 'default', 'balance', 'housing', 'loan', 'day', 'duration', 'campaign', 'pdays', 'previous', 'y', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_divorced', 'marital_married', 'marital_single', 'education_primary', 'education_secondary', 'education_tertiary', 'education_unknown', 'contact_cellular', 'contact_telephone', 'contact_unknown', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_failure', 'poutcome_other', 'poutcome_success', 'poutcome_unknown']


## 🎯 Step 5: Split Features & Target

Separating the dataset into:
- X: Features (input)  → all columns except 'y'
- y: Target  (output)  → 'y' column (0=No, 1=Yes)

We also remove 'duration' from features because
it directly affects the outcome and is not available
before the call is made in real-world scenarios.

In [37]:
# Split Features & Target
X = df.drop(['y', 'duration'], axis=1)
y = df['y']

print("✅ Split done successfully!")
print("Features Shape (X):", X.shape)
print("Target Shape   (y):", y.shape)
print("\nTarget Distribution:")
print(y.value_counts())
print("\nPercentage:")
print(round(y.value_counts(normalize=True) * 100, 2))

✅ Split done successfully!
Features Shape (X): (45208, 47)
Target Shape   (y): (45208,)

Target Distribution:
y
0    39919
1     5289
Name: count, dtype: int64

Percentage:
y
0    88.3
1    11.7
Name: proportion, dtype: float64


## ✂️ Step 6: Train-Test Split

Splitting the dataset into:
- Training set: 80% → used to train the model
- Testing set:  20% → used to evaluate the model

We split BEFORE applying SMOTE to avoid data leakage.

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Split done successfully!")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("\nTrain Target Distribution:")
print(round(y_train.value_counts(normalize=True) * 100, 2))
print("\nTest Target Distribution:")
print(round(y_test.value_counts(normalize=True) * 100, 2))

✅ Split done successfully!
X_train: (36166, 47)
X_test : (9042, 47)
y_train: (36166,)
y_test : (9042,)

Train Target Distribution:
y
0    88.3
1    11.7
Name: proportion, dtype: float64

Test Target Distribution:
y
0    88.3
1    11.7
Name: proportion, dtype: float64


## ⚖️ Step 7: Scaling Numerical Features

Scaling numerical features so that all values
are on the same scale.

We use StandardScaler which transforms values to:
- Mean = 0
- Standard Deviation = 1

We fit ONLY on training data to avoid data leakage,
then transform both train and test sets.

In [39]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Numerical columns to scale
num_cols = ['age', 'balance', 'day', 'campaign',
            'pdays', 'previous']

# Fit on train, transform both
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print("✅ Scaling done successfully!")
print("\nAfter Scaling - X_train sample:")
print(X_train[num_cols].describe().round(2))

✅ Scaling done successfully!

After Scaling - X_train sample:
            age   balance       day  campaign     pdays  previous
count  36166.00  36166.00  36166.00  36166.00  36166.00  36166.00
mean       0.00     -0.00      0.00      0.00     -0.00     -0.00
std        1.00      1.00      1.00      1.00      1.00      1.00
min       -2.15     -3.03     -1.78     -0.57     -0.41     -0.24
25%       -0.74     -0.42     -0.94     -0.57     -0.41     -0.24
50%       -0.18     -0.30      0.02     -0.25     -0.41     -0.24
75%        0.67      0.02      0.62      0.08     -0.41     -0.24
max        5.09     32.56      1.82     19.60      8.30    114.73


## 🔁 Step 8: Handle Imbalanced Data using SMOTE

The dataset is highly imbalanced:
- No  (0): 88.3%
- Yes (1): 11.7%

We use SMOTE (Synthetic Minority Over-sampling Technique)
to generate synthetic samples for the minority class.

⚠️ SMOTE is applied ONLY on training data
   to avoid data leakage.

In [40]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("✅ SMOTE applied successfully!")
print("\nBefore SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_sm.value_counts())
print("\nPercentage After SMOTE:")
print(round(y_train_sm.value_counts(normalize=True) * 100, 2))

✅ SMOTE applied successfully!

Before SMOTE:
y
0    31935
1     4231
Name: count, dtype: int64

After SMOTE:
y
0    31935
1    31935
Name: count, dtype: int64

Percentage After SMOTE:
y
0    50.0
1    50.0
Name: proportion, dtype: float64


## 💾 Step 9: Export Preprocessed Data

Exporting the preprocessed data as CSV files
so we can use them directly in the modeling notebook.

- train.csv → Training data (after SMOTE)
- test.csv  → Testing data  (original)

In [41]:
# Combine X and y
train_data = X_train_sm.copy()
train_data['y'] = y_train_sm.values

test_data = X_test.copy()
test_data['y'] = y_test.values

# Export to CSV
train_data.to_csv('train.csv', index=False)
test_data.to_csv('test.csv', index=False)

print("✅ Data exported successfully!")
print("train.csv shape:", train_data.shape)
print("test.csv shape :", test_data.shape)

✅ Data exported successfully!
train.csv shape: (63870, 48)
test.csv shape : (9042, 48)
